#### Transform Races Data (Bronze to Silver)

**Steps:**
1. Read the raw data from the Bronze table
2. Drop the `url` column
3. Rename `raceName` to `race_name` and `circuitId` to `circuit_id`
4. Remove duplicates based on `season` and `round`
5. Apply title case to `race_name`
6. Save to Silver table

#### Loading Configuration

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

#### Setting Variables
Define the Bronze source table and Silver target table names.

#### Read Bronze Table
Read the raw races data from `formula1.bronze.races` into a DataFrame.

In [0]:
races_df = spark.read.table(bronze_table)
# or circuits_df = spark.table(bronze_table) best is to use read.table as it is more explicit
display(races_df)

#### Drop Columns
Remove the `url` column - not needed for analysis. Keep only the selected columns.

In [0]:
from pyspark.sql.functions import *
races_selected_df = races_df.select(
    col('season'),
    col('round'),
    col('raceName'),
    col('date'),
    col('circuitId'),
    col('ingestion_timestamp'),
    col('source_file')
   )

In [0]:
display(races_selected_df)

#### Rename Columns
Rename `circuitID` to `circuit_id` and `raceName` to `race_name` using `withColumnsRenamed()`.

In [0]:
races_renamed_df = (races_selected_df
 .withColumnsRenamed({
   'circuitID':'circuit_id',                                          
   'raceName':'race_name',
   'date':'race_date'                                          
  })) # this uses withcolumn(S)

#### Remove Duplicates
Drop duplicate rows based on `season` and `round` columns.

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(['season','round'])

In [0]:

display(races_distinct_df)

#### Title Case
Convert `race_name` to title case using `initcap()` (e.g., "british grand prix" becomes "British Grand Prix").

In [0]:
from pyspark.sql.functions import initcap
races_final_df = (races_distinct_df
                  .withColumn('race_name', initcap(col("race_name"))))


In [0]:
display(races_final_df)


#### Write to Silver Table
Save the cleaned DataFrame to `formula1.silver.races` in Delta format with overwrite mode.

In [0]:
(
  races_final_df
  .write
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .format("delta")
  .saveAsTable(f'{catalog_name}.{silver_schema}.races')
)

In [0]:
spark.read.table(silver_schema).display()